# 03 — Net PnL per Trade

Computes net PnL using V4 formula:
`net_pnl = fill_size × |spread| - fill_size × (fee_bps + slippage_bps) / 10000`

Config loaded from `backtest.config.json` (V9, C3). No hardcoded constants.

**Invariants**: V4, V7 (net_edge > 0 required before claiming profit).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import config

cfg = config.load()
fee_bps       = cfg['fee_bps']
slippage_bps  = cfg['slippage_bps']
cost_rate     = (fee_bps + slippage_bps) / 10_000
print(f'fee_bps={fee_bps}, slippage_bps={slippage_bps}, cost_rate={cost_rate}')

In [ ]:
joined = pd.read_parquet('../data/raw/joined.parquet')

# Only filled trades have PnL (fill_price and fill_size both present)
filled = joined.dropna(subset=['fill_price', 'fill_size']).copy()
filled['fill_size']  = filled['fill_size'].astype(float)
filled['spread']     = filled['spread'].astype(float)

# V4 formula
filled['gross_pnl'] = filled['fill_size'] * filled['spread'].abs()
filled['cost']      = filled['fill_size'] * cost_rate
filled['net_pnl']   = filled['gross_pnl'] - filled['cost']

print(f'Filled trades: {len(filled)}')
print(filled[['correlation_id','spread','fill_size','gross_pnl','cost','net_pnl']].head())

In [ ]:
# V7: any claim of profit must cite net_pnl > 0, not gross
profitable = filled['net_pnl'] > 0
print(f'Profitable fills (net): {profitable.sum()} / {len(filled)}')
print(f'Total net PnL: {filled["net_pnl"].sum():.6f}')
print(f'Mean net PnL per trade: {filled["net_pnl"].mean():.6f}')

In [ ]:
filled.to_parquet('../data/raw/filled_pnl.parquet', index=False)
print('Saved data/raw/filled_pnl.parquet')